In [33]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.linalg import svd

import jlinops

In [3]:
# Generate random problem
n = 100
m = 60
k = 75

np.random.seed(0)
A = np.random.normal(size=(m,n))
L = np.random.normal(size=(k,n))
xtrue = np.random.normal(size=n)
btrue = A @ xtrue
noise_sigma = 0.01
b = btrue + noise_sigma*np.random.normal(size=len(btrue))

In [7]:
# Check if the condition is met?
eta = 1.01
tau = 1.0
e_norm_sq = m*(noise_sigma**2)
check_one = tau < ( np.linalg.norm(b)**2 ) - (eta**2)*e_norm_sq
# check_two = 
# b0 = 

In [8]:
b0 = b - A @ (np.linalg.pinv(A) @ b)

In [9]:
( np.linalg.norm(b)**2 ) - (eta**2)*e_norm_sq

4866.278483660476

In [10]:
( np.linalg.norm(b0)**2 ) - (eta**2)*e_norm_sq

-0.0061206

In [12]:
tmp, _, _, _ = np.linalg.lstsq(A, b)
b2 = b - A @ tmp

/var/folders/fn/xbs7g2ln57x3xrdggzf_dzzm0000gr/T/ipykernel_7702/934401818.py:1: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  tmp, _, _, _ = np.linalg.lstsq(A, b)


# Put it all together

In [19]:
from scipy.linalg import null_space

In [24]:
U, _ = jlinops.first_order_derivative_1d(10, boundary="zero")
U = U.toarray()

In [27]:
W = null_space(L)

In [28]:
W.shape

(100, 25)

In [30]:
(A @ W).shape

(60, 25)

In [ ]:
def discrepancy_principle_reichel_shyshkov(A, L, b, noise_var, eta=1.025, b0=None, maxits=25):
    """Selects a regularization parameter using the method of Reichel and Shyshkov (2008),
    which uses a 3rd order root-finder to satisfy the discrepancy principle. The method is guaranteed 
    to converge (the code will throw an error if not guaranteed).

    The code assumes that the problem is NOT already in standard form. Then, to use the method, the problem
    is placed into standard form.
    """

    # Whiten A and b
    A = (1.0/np.sqrt(noise_var))*A
    b = (1.0/np.sqrt(noise_var))*b
    bnorm = np.linalg.norm(b)
    
    # Place problem into standard form?
    W = null_space(L)
    if W.shape[1] == 0:
        Abar = A
        resid_norm_xker = 0.0
        xker = np.zeros(A.shape[0])

    else:
        Lpinv = np.linalg.pinv(L)
        AWpinv = np.linalg.pinv(A @ W)
        Lopinv = ( np.eye(A.shape[1]) - W @ (AWpinv @ A) ) @ Lpinv
        Abar = A @ Lopinv
        xker = W @ (AWpinv @ b)
        resid_norm_xker = np.linalg.norm(A @ xker - b)

    # Compute orthogonal projection of b onto null A.
    if b0 is None:
        tmp, _, _, _ = np.linalg.lsqsq(Abar, b)
        b0 = b - Abar @ tmp
    else:
        pass

    # Set varepsilon = len(b)*noise_var
    varepsilon = len(b)*noise_var
   
    # Is || b_0 || < \eta \varepsilon < || b || ?
    check1 = np.linalg.norm(b0) < eta*varepsilon
    check2 = eta*varepsilon < np.linalg.norm(b)
    assert (check1 and check2), "not guaranteed to have a positive zero (||b0|| is very large)"
    
    
    # Build functions for \phi, \phi', and \phi''

    def phi(beta):
        return None
    
    def phi_prime(beta):
        return None
    
    def phi_prime_prime(beta):
        return None




In [32]:
import numpy as np
from scipy.linalg import svd

# Create a random rectangular matrix A
A = np.random.rand(4, 3)

# Compute the SVD
U, S, Vt = svd(A)

# Construct the bidiagonal matrix B
B = np.zeros_like(A, dtype=float)
np.fill_diagonal(B, S)

print("Matrix A:")
print(A)
print("\nOrthogonal Matrix U:")
print(U)
print("\nBidiagonal Matrix B:")
print(B)
print("\nOrthogonal Matrix V^T:")
print(Vt)

Matrix A:
[[0.23653069 0.59809695 0.71794888]
 [0.19649304 0.00095644 0.21976466]
 [0.94970792 0.44189585 0.93021941]
 [0.95337358 0.63233459 0.0618019 ]]

Orthogonal Matrix U:
[[-0.44131416  0.53648088  0.70702287 -0.13247167]
 [-0.13416951  0.09537758 -0.33026026 -0.92942446]
 [-0.71982642  0.23364643 -0.56501456  0.32866064]
 [-0.51873912 -0.80529543  0.26796525 -0.10297386]]

Bidiagonal Matrix B:
[[1.90787372 0.         0.        ]
 [0.         0.70464999 0.        ]
 [0.         0.         0.39322188]
 [0.         0.         0.        ]]

Orthogonal Matrix V^T:
[[-0.68606467 -0.47706615 -0.54929333]
 [-0.56796468 -0.12064141  0.81416324]
 [-0.45467725  0.87054784 -0.18818891]]
